# M6 PCB 瑕疵偵測訓練（Colab，GPU 版）

本機 Mac 用 CPU/MPS 也能訓練（見專案 `CLAUDE.md` 的實測時間），這份 notebook 是 GPU 版本，給需要更快跑更多 epoch、或沒有 Mac 的人用。

資料集：[DeepPCB](https://github.com/tangsanli5201/DeepPCB)，MIT License，1500 張 640x640 PCB 瑕疵標註圖。

使用前：**Runtime → Change runtime type → T4 GPU**。

In [ ]:
!pip install -q ultralytics==8.4.160

## 1. 下載資料集與轉換腳本

直接 clone DeepPCB 原始資料，以及這個專案（重用 `scripts/prepare_deeppcb.py` 的轉換邏輯，跟本機訓練用同一份程式碼，結果才有可比性）。

In [ ]:
!git clone --depth 1 https://github.com/tangsanli5201/DeepPCB.git /content/DeepPCB
!git clone --depth 1 https://github.com/thothawei/vision-ai-demo.git /content/vision-ai-demo

In [ ]:
import sys
sys.path.insert(0, "/content/vision-ai-demo")

# 用專案裡的轉換腳本，但把路徑指到 Colab 的資料位置
import importlib.util
spec = importlib.util.spec_from_file_location("prepare_deeppcb", "/content/vision-ai-demo/scripts/prepare_deeppcb.py")
prepare_deeppcb = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prepare_deeppcb)

prepare_deeppcb.RAW_DIR = __import__("pathlib").Path("/content/DeepPCB/PCBData")
prepare_deeppcb.OUT_DIR = __import__("pathlib").Path("/content/deeppcb_yolo")
prepare_deeppcb.main()

## 2. 訓練

GPU 上可以跑更多 epoch、更大 batch。本機 Mac（M1 Pro，MPS）60 epoch 實測時間見 `CLAUDE.md`；GPU 應該明顯更快，可以把 epoch 數往上調。

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(
    data="/content/deeppcb_yolo/data.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    patience=20,
    project="/content/runs",
    name="pcb_yolo11n",
    seed=42,
)

## 3. 在官方測試集上驗證（跟本機訓練用同一個測試集切分）

In [ ]:
metrics = model.val(data="/content/deeppcb_yolo/data.yaml", split="test")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## 4. 下載權重，放回本機專案的 `models/defect/pcb_yolo11n/weights/best.pt`

In [ ]:
from google.colab import files
files.download("/content/runs/pcb_yolo11n/weights/best.pt")